Import important library like pandas, regex, and json

In [83]:
import pandas as pd
import numpy as np
import re
import json

In [84]:
df = pd.read_csv("../../data/laptop_mentah.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            33 non-null     int64
 1   name          33 non-null     str  
 2   price         33 non-null     str  
 3   imageUrl      33 non-null     str  
 4   rating        33 non-null     int64
 5   countReview   33 non-null     int64
 6   shop          33 non-null     str  
 7   product_link  33 non-null     str  
dtypes: int64(3), str(5)
memory usage: 2.2 KB


See what the data looks like

In [85]:
df.head()

,id,name,price,imageUrl,rating,countReview,shop,product_link
0,103833384568,Acer Nitro V 16S AI ANV16S-41-R70Y RTX 5060 8G...,Rp24.374.999,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 2153004, 'name': 'Acer Official Authori...",https://tokopedia.com103833384568
1,103684473901,[CO] ACER NITRO V 16S AI ANV16S GeForce RTX 50...,Rp24.032.131,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 10002818, 'name': 'NVIDIA GeForce Lapto...",https://tokopedia.com103684473901
2,102055980121,[ FREE HEADSET ] ACER NITRO V 16S AI ANV16S Ge...,Rp23.363.010,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 10002818, 'name': 'NVIDIA GeForce Lapto...",https://tokopedia.com102055980121
3,103741157994,ACER NITRO V 16S GeForce RTX 5060 8GB - RYZEN ...,Rp24.360.204,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 16126198, 'name': 'NVIDIA GeForce Offic...",https://tokopedia.com103741157994
4,102777889635,Acer Nitro V 16S RTX 5060 Ryzen 7 260 32GB 1TB...,Rp23.869.000,https://p19-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 2185542, 'name': 'ROGZ Online', 'city':...",https://tokopedia.com102777889635


No ram nor storage? no problem. we'll fix it

In [86]:
# 1. GET NILAI KAPASITAS STORAGE
def ekstrak_kapasitas_storage(judul):
    judul_pola = str(judul).lower()
    # Mencari angka populer (250, 256, 512, 1, 2) yang diikuti oleh GB atau TB
    pola_kapasitas = r'(250|256|512|1|2)\s*(gb|tb)'
    hasil = re.search(pola_kapasitas, judul_pola)
    
    if hasil:
        angka = hasil.group(1)
        satuan = hasil.group(2)
        # Jika satuannya TB (terabyte), kita konversi nilainya ke GB (dikali 1000) agar setara
        if satuan == 'tb':
            return int(angka) * 1000
        return int(angka)
    return None

# 2. TERAPKAN EKSTRAKSI NILAI STORAGE
df['Storage_Capacity_GB'] = df['name'].apply(ekstrak_kapasitas_storage)

# 3. DATA IMPUTATION: Jika nilai storage ada, otomatis kunci tipe storagenya sebagai 'SSD'
# Ini menyelamatkan data laptop yang tipenya sempat terbaca 'Unknown' di tabel Anda sebelumnya
df.loc[df['Storage_Capacity_GB'].notna(), 'Storage_Type'] = 'SSD'

#=========================================RAM==========================
# Pola: (\d+)\s*(?:GB|gb|Gb) -> Mencari angka (\d+), abaikan spasi \s*, lalu cari kata GB
def ekstrak_ram(judul):
    pola_ram = r'(\d+)\s*(?:GB|gb|Gb)'
    hasil = re.search(pola_ram, str(judul))
    if hasil:
        return int(hasil.group(1)) # Mengambil angkanya saja (misal: 8 atau 16)
    return None

df['RAM_GB'] = df['name'].apply(ekstrak_ram)

df.head()

,id,name,price,imageUrl,rating,countReview,shop,product_link,Storage_Capacity_GB,Storage_Type,RAM_GB
0,103833384568,Acer Nitro V 16S AI ANV16S-41-R70Y RTX 5060 8G...,Rp24.374.999,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 2153004, 'name': 'Acer Official Authori...",https://tokopedia.com103833384568,512.0,SSD,8.0
1,103684473901,[CO] ACER NITRO V 16S AI ANV16S GeForce RTX 50...,Rp24.032.131,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 10002818, 'name': 'NVIDIA GeForce Lapto...",https://tokopedia.com103684473901,512.0,SSD,16.0
2,102055980121,[ FREE HEADSET ] ACER NITRO V 16S AI ANV16S Ge...,Rp23.363.010,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 10002818, 'name': 'NVIDIA GeForce Lapto...",https://tokopedia.com102055980121,512.0,SSD,16.0
3,103741157994,ACER NITRO V 16S GeForce RTX 5060 8GB - RYZEN ...,Rp24.360.204,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 16126198, 'name': 'NVIDIA GeForce Offic...",https://tokopedia.com103741157994,512.0,SSD,8.0
4,102777889635,Acer Nitro V 16S RTX 5060 Ryzen 7 260 32GB 1TB...,Rp23.869.000,https://p19-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 2185542, 'name': 'ROGZ Online', 'city':...",https://tokopedia.com102777889635,2.0,SSD,32.0


the links look weird.

In [87]:
df['product_link'] = "https://tokopedia.com" + '/i/' + df['id'].astype(str)
df.head()

,id,name,price,imageUrl,rating,countReview,shop,product_link,Storage_Capacity_GB,Storage_Type,RAM_GB
0,103833384568,Acer Nitro V 16S AI ANV16S-41-R70Y RTX 5060 8G...,Rp24.374.999,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 2153004, 'name': 'Acer Official Authori...",https://tokopedia.com/i/103833384568,512.0,SSD,8.0
1,103684473901,[CO] ACER NITRO V 16S AI ANV16S GeForce RTX 50...,Rp24.032.131,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 10002818, 'name': 'NVIDIA GeForce Lapto...",https://tokopedia.com/i/103684473901,512.0,SSD,16.0
2,102055980121,[ FREE HEADSET ] ACER NITRO V 16S AI ANV16S Ge...,Rp23.363.010,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 10002818, 'name': 'NVIDIA GeForce Lapto...",https://tokopedia.com/i/102055980121,512.0,SSD,16.0
3,103741157994,ACER NITRO V 16S GeForce RTX 5060 8GB - RYZEN ...,Rp24.360.204,https://p16-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 16126198, 'name': 'NVIDIA GeForce Offic...",https://tokopedia.com/i/103741157994,512.0,SSD,8.0
4,102777889635,Acer Nitro V 16S RTX 5060 Ryzen 7 260 32GB 1TB...,Rp23.869.000,https://p19-images-sign-sg.tokopedia-static.ne...,0,0,"{'id': 2185542, 'name': 'ROGZ Online', 'city':...",https://tokopedia.com/i/102777889635,2.0,SSD,32.0


looks nice, but i want to sort it by price

In [88]:
if df['price'].dtype == 'O':
    df['price'] = df['price'].str.replace('Rp', '', regex=False)
    df['price'] = df['price'].str.replace('.', '', regex=False)
    df['price'] = df['price'].str.strip()
    df['price'] = df['price'].astype(int)

df_cheapest = df.sort_values(by='price', ascending=True).head(10)
df_cheapest[['name', 'price', 'RAM_GB', 'Storage_Capacity_GB']]

,name,price,RAM_GB,Storage_Capacity_GB
30,Laptop RARE RTX 4060 Victus 16 BY HP GAMING R7...,Rp15.999.000,NaN,NaN
25,Laptop RARE RTX 4070 Victus 16 S1777AX BY HP G...,Rp18.999.000,NaN,NaN
18,Acer Gaming NITRO V16S - AMD Ryzen 7 260 RTX 5...,Rp22.249.000,8.0,512.0
8,[BEST SELLER] Acer Gaming NITRO V16S - AMD Ryz...,Rp22.498.686,8.0,512.0
5,ACER NITRO V 16S ANV16S-41-R70Y RYZEN 7 260 16...,Rp22.599.000,16.0,512.0
2,[ FREE HEADSET ] ACER NITRO V 16S AI ANV16S Ge...,Rp23.363.010,16.0,512.0
19,Laptop Acer Nitro V 16s AI ANV16S Ryzen 7-260 ...,Rp23.869.000,16.0,512.0
4,Acer Nitro V 16S RTX 5060 Ryzen 7 260 32GB 1TB...,Rp23.869.000,32.0,2.0
15,Acer Nitro V 16S AI Gaming Laptop | AMD Ryzen ...,Rp23.869.000,16.0,2.0
12,ACER NITRO V 16S RYZEN 7 260 RTX 5060 8GB RAM ...,Rp23.869.000,8.0,512.0


Nan? Unknown? no Problem

In [91]:
# 1. Hapus baris yang RAM-nya NaN langsung di tabel utama 'df'
df.dropna(subset=['RAM_GB'], inplace=True)

# 2. Hapus baris yang Storage_Type-nya 'Unknown' langsung di tabel utama 'df'
df = df[df['Storage_Type'] != 'Unknown']

# 3. Urutan termurah baru dari variabel 'df' yang SUDAH BERSIH
df_bersih_terurut = df.sort_values(by='price', ascending=True).head(10)

# 4. Tampilkan tabel hasil akhir yang sudah steril 100% dari NaN
df_bersih_terurut[['name', 'price', 'RAM_GB', 'Storage_Type', 'Storage_Capacity_GB']]

,name,price,RAM_GB,Storage_Type,Storage_Capacity_GB
18,Acer Gaming NITRO V16S - AMD Ryzen 7 260 RTX 5...,Rp22.249.000,8.0,SSD,512.0
8,[BEST SELLER] Acer Gaming NITRO V16S - AMD Ryz...,Rp22.498.686,8.0,SSD,512.0
5,ACER NITRO V 16S ANV16S-41-R70Y RYZEN 7 260 16...,Rp22.599.000,16.0,SSD,512.0
2,[ FREE HEADSET ] ACER NITRO V 16S AI ANV16S Ge...,Rp23.363.010,16.0,SSD,512.0
15,Acer Nitro V 16S AI Gaming Laptop | AMD Ryzen ...,Rp23.869.000,16.0,SSD,2.0
4,Acer Nitro V 16S RTX 5060 Ryzen 7 260 32GB 1TB...,Rp23.869.000,32.0,SSD,2.0
19,Laptop Acer Nitro V 16s AI ANV16S Ryzen 7-260 ...,Rp23.869.000,16.0,SSD,512.0
11,Acer Nitro V16S Ryzen 7-260 RTX-5060 8GB/ 16GB...,Rp23.869.000,8.0,SSD,512.0
12,ACER NITRO V 16S RYZEN 7 260 RTX 5060 8GB RAM ...,Rp23.869.000,8.0,SSD,512.0
17,[XBP] Acer Nitro V 16S AI ANV16S-41-R70Y (NH.Q...,Rp23.979.899,16.0,SSD,512.0
